In [1]:
import pandas as pd
import numpy as np
import re 
import warnings
warnings.filterwarnings('ignore')

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse as sp 
import pickle

In [2]:
nltk.download('stopwords', quiet = True)
nltk.download('punkt_tab', quiet = True)
nltk.download('wordnet', quiet = True)


True

In [3]:
df = pd.read_csv("C:/Users/dell/OneDrive/Desktop/Project NLP/Tweets.csv")
print(f' Shape: {df.shape}')
print(f'Tweets: {df.shape[0]:,}')
print(f' Features: {df.shape[1]}')


 Shape: (14640, 15)
Tweets: 14,640
 Features: 15


DATA CLEANING

In [4]:
# Xóa cột không cần thiết
columns_to_drop = [
    'airline_sentiment_gold',
    'negativereason_gold',
    'tweet_coord',
    'tweet_location',
    'user_timezone',
    'name',
    'retweet_count',
    'tweet_id'
]
df_clean = df.drop(columns = columns_to_drop, errors = 'ignore')


HANDLE MISSING VALUES

In [5]:
df_clean.isna().sum()


airline_sentiment                  0
airline_sentiment_confidence       0
negativereason                  5462
negativereason_confidence       4118
airline                            0
text                               0
tweet_created                      0
dtype: int64

In [6]:
df_clean['negativereason'] = df_clean['negativereason'].fillna('Not Applicable')
df_clean['negativereason_confidence'] = df_clean['negativereason_confidence'].fillna(0.0)
df_clean.shape
df_clean.isna().sum()

airline_sentiment               0
airline_sentiment_confidence    0
negativereason                  0
negativereason_confidence       0
airline                         0
text                            0
tweet_created                   0
dtype: int64

TEXT CLEANING

In [7]:
def clean_text(text):
    #Lowercase
    text = text.lower()
    #Remove URLs
    text = re.sub(r'http\S+\www\S+|https\S+', '', text)
    #Remove mentions
    text = re.sub(r'@\w+', '', text)
    # Remove hashtags
    text = re.sub(r'#\w+', '', text)
    # Remove special chars and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text
# Test
for i in range(3):
    original = df_clean['text'].iloc[i]
    cleaned = clean_text(original)
    print(f'BEFORE: {original}')
    print(f'AFTER: {cleaned}')

BEFORE: @VirginAmerica What @dhepburn said.
AFTER: what said
BEFORE: @VirginAmerica plus you've added commercials to the experience... tacky.
AFTER: plus youve added commercials to the experience tacky
BEFORE: @VirginAmerica I didn't today... Must mean I need to take another trip!
AFTER: i didnt today must mean i need to take another trip


In [8]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english')) - {'no', 'not', ' never'}

def preprocess_text (text):
    text = clean_text(text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words and len (t) > 2]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

df_clean['text_processed'] = df_clean['text'].apply(preprocess_text)
print(f'{len(df_clean):,} tweets đã xử lý')

14,640 tweets đã xử lý


In [9]:
df_clean['label'] = df_clean['airline_sentiment'].map(
    {'negative': 0, 'neutral': 1, 'positive': 2}
)

# Split
X_train, X_temp, y_train, y_temp = train_test_split(
    df_clean['text_processed'], df_clean['label'],
    test_size=0.3, random_state=42, stratify=df_clean['label']
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5, random_state=42, stratify=y_temp
)

# TF-IDF
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2), min_df=2, sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf   = tfidf.transform(X_val)
X_test_tfidf  = tfidf.transform(X_test)

sp.save_npz('X_train_tfidf.npz', X_train_tfidf)
sp.save_npz('X_val_tfidf.npz', X_val_tfidf)
sp.save_npz('X_test_tfidf.npz', X_test_tfidf)

y_train.to_csv('y_train.csv', index=False)
y_val.to_csv('y_val.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

pickle.dump(tfidf, open('tfidf_vectorizer.pkl', 'wb'))
